# Exercise 01 — CUDA Environment Setup

In [ ]:
!nvidia-smi
!nvcc --version
!nvidia-smi --query-gpu=name,compute_cap --format=csv

Tue Oct 13 09:29:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%writefile hello.cu
#include <cstdio>
#include <cuda_runtime.h>

__global__ void hello()
{
    printf("Hello from GPU thread %d\n", threadIdx.x);
}

int main()
{
    hello<<<1, 4>>>();

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        fprintf(stderr, "Kernel launch failed: %s\n",
                cudaGetErrorString(err));
        return 1;
    }

    err = cudaDeviceSynchronize();
    if (err != cudaSuccess) {
        fprintf(stderr, "GPU execution failed: %s\n",
                cudaGetErrorString(err));
        return 1;
    }

    return 0;
}

Writing hello.cu


In [ ]:
!nvcc -arch=sm_75 hello.cu -o hello && ./hello

Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3


# Exercise 02 — Adding Two Numbers

In [ ]:
%%writefile ex2_add.cu
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do {                                      \
    cudaError_t err = (call);                                      \
    if (err != cudaSuccess) {                                     \
        fprintf(stderr, "CUDA error at line %d: %s\n",             \
                __LINE__, cudaGetErrorString(err));                \
        exit(EXIT_FAILURE);                                       \
    }                                                             \
} while (0)

__global__ void add(const int *a, const int *b, int *c)
{
    *c = *a + *b;
}

int main()
{
    int a = 2;
    int b = 7;
    int c = 0;
    int *d_a, *d_b, *d_c;

    CUDA_CHECK(cudaMalloc((void **)&d_a, sizeof(int)));
    CUDA_CHECK(cudaMalloc((void **)&d_b, sizeof(int)));
    CUDA_CHECK(cudaMalloc((void **)&d_c, sizeof(int)));

    CUDA_CHECK(cudaMemcpy(d_a, &a, sizeof(int),
                          cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, &b, sizeof(int),
                          cudaMemcpyHostToDevice));

    add<<<1, 1>>>(d_a, d_b, d_c);
    CUDA_CHECK(cudaGetLastError());

    CUDA_CHECK(cudaMemcpy(&c, d_c, sizeof(int),
                          cudaMemcpyDeviceToHost));

    printf("Result is %d\n", c);
    printf("Validation: %s\n", c == a + b ? "CORRECT" : "INCORRECT");

    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));

    return c == a + b ? EXIT_SUCCESS : EXIT_FAILURE;
}

Writing ex2_add.cu


In [ ]:
!nvcc -arch=sm_75 ex2_add.cu -o ex2_add && ./ex2_add

Result is 9
Validation: CORRECT


# Exercise 03 — Vector Addition with 512 Blocks

In [ ]:
%%writefile ex3_blocks.cu
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define N 512

#define CUDA_CHECK(call) do {                                      \
    cudaError_t err = (call);                                      \
    if (err != cudaSuccess) {                                     \
        fprintf(stderr, "CUDA error at line %d: %s\n",             \
                __LINE__, cudaGetErrorString(err));                \
        exit(EXIT_FAILURE);                                       \
    }                                                             \
} while (0)

__global__ void add(const int *a, const int *b, int *c)
{
    int i = blockIdx.x;
    c[i] = a[i] + b[i];
}

void random_ints(int *x, int count)
{
    for (int i = 0; i < count; ++i)
        x[i] = rand() % 100;
}

int main()
{
    size_t bytes = N * sizeof(int);

    int *a = (int *)malloc(bytes);
    int *b = (int *)malloc(bytes);
    int *c = (int *)malloc(bytes);

    if (!a || !b || !c) {
        fprintf(stderr, "Host memory allocation failed.\n");
        free(a);
        free(b);
        free(c);
        return EXIT_FAILURE;
    }

    random_ints(a, N);
    random_ints(b, N);

    int *d_a, *d_b, *d_c;

    CUDA_CHECK(cudaMalloc((void **)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_c, bytes));

    CUDA_CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));

    add<<<N, 1>>>(d_a, d_b, d_c);
    CUDA_CHECK(cudaGetLastError());

    CUDA_CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));

    int errors = 0;

    for (int i = 0; i < N; ++i) {
        printf("%d) %d + %d = %d\n", i, a[i], b[i], c[i]);

        if (c[i] != a[i] + b[i])
            ++errors;
    }

    printf("Launch: %d blocks x 1 thread = %d threads\n", N, N);
    printf("Verification: %d mismatches out of %d\n", errors, N);

    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));

    free(a);
    free(b);
    free(c);

    return errors == 0 ? EXIT_SUCCESS : EXIT_FAILURE;
}

Writing ex3_blocks.cu


In [ ]:
!nvcc -arch=sm_75 ex3_blocks.cu -o ex3_blocks && ./ex3_blocks > ex3_output.txt && tail -n 12 ex3_output.txt

502) 60 + 52 = 112
503) 17 + 50 = 67
504) 23 + 62 = 85
505) 61 + 82 = 143
506) 81 + 99 = 180
507) 9 + 17 = 26
508) 90 + 54 = 144
509) 25 + 73 = 98
510) 96 + 15 = 111
511) 67 + 6 = 73
Launch: 512 blocks x 1 thread = 512 threads
Verification: 0 mismatches out of 512


# Exercise 04 — Vector Addition with 512 Threads

In [ ]:
from pathlib import Path

code = Path("ex3_blocks.cu").read_text()
code = code.replace("int i = blockIdx.x;", "int i = threadIdx.x;")
code = code.replace("add<<<N, 1>>>", "add<<<1, N>>>")
code = code.replace(
    'printf("Launch: %d blocks x 1 thread = %d threads\\n", N, N);',
    'printf("Launch: 1 block x %d threads = %d threads\\n", N, N);'
)

Path("ex4_threads.cu").write_text(code)
print("Created ex4_threads.cu")

Created ex4_threads.cu


In [ ]:
!nvcc -arch=sm_75 ex4_threads.cu -o ex4_threads && ./ex4_threads > ex4_output.txt && tail -n 12 ex4_output.txt

502) 60 + 52 = 112
503) 17 + 50 = 67
504) 23 + 62 = 85
505) 61 + 82 = 143
506) 81 + 99 = 180
507) 9 + 17 = 26
508) 90 + 54 = 144
509) 25 + 73 = 98
510) 96 + 15 = 111
511) 67 + 6 = 73
Launch: 1 block x 512 threads = 512 threads
Verification: 0 mismatches out of 512


In [ ]:
from pathlib import Path

code = Path("ex4_threads.cu").read_text()
code = code.replace("#define N 512", "#define N 2048")
Path("ex4_limit.cu").write_text(code)

print("Created ex4_limit.cu with 2048 threads per block")

Created ex4_limit.cu with 2048 threads per block


In [ ]:
!nvcc -arch=sm_75 ex4_limit.cu -o ex4_limit && (./ex4_limit > ex4_limit_output.txt 2>&1; result=$?; cat ex4_limit_output.txt; echo "Program exit code: $result")

CUDA error at line 57: invalid argument
Program exit code: 1


# Exercise 05 — Vector Multiplication

In [ ]:
%%writefile ex5_vecmul.cu
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define N 10000000
#define THREADS_PER_BLOCK 512

#define CUDA_CHECK(call) do {                                      \
    cudaError_t err = (call);                                      \
    if (err != cudaSuccess) {                                     \
        fprintf(stderr, "CUDA error at line %d: %s\n",             \
                __LINE__, cudaGetErrorString(err));                \
        exit(EXIT_FAILURE);                                       \
    }                                                             \
} while (0)

__global__ void vecMul(const int *a, const int *b, int *c, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n)
        c[i] = a[i] * b[i];
}

void random_ints(int *x, int count)
{
    for (int i = 0; i < count; ++i)
        x[i] = rand() % 100;
}

int main()
{
    size_t bytes = (size_t)N * sizeof(int);

    int *a = (int *)malloc(bytes);
    int *b = (int *)malloc(bytes);
    int *c = (int *)malloc(bytes);

    if (!a || !b || !c) {
        fprintf(stderr, "Host allocation failed.\n");
        free(a);
        free(b);
        free(c);
        return EXIT_FAILURE;
    }

    random_ints(a, N);
    random_ints(b, N);

    int *d_a, *d_b, *d_c;

    CUDA_CHECK(cudaMalloc((void **)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_c, bytes));

    CUDA_CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));

    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;

    printf("Launching %d blocks x %d threads\n",
           blocks, THREADS_PER_BLOCK);

    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    CUDA_CHECK(cudaGetLastError());

    CUDA_CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));

    for (int i = N - 1000; i < N; ++i)
        printf("%d) %d x %d = %d\n", i, a[i], b[i], c[i]);

    int errors = 0;
    for (int i = 0; i < N; ++i) {
        if (c[i] != a[i] * b[i])
            ++errors;
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);

    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));

    free(a);
    free(b);
    free(c);

    return errors == 0 ? EXIT_SUCCESS : EXIT_FAILURE;
}

Writing ex5_vecmul.cu


In [ ]:
!nvcc -arch=sm_75 ex5_vecmul.cu -o ex5_vecmul && ./ex5_vecmul > ex5_output.txt && head -n 2 ex5_output.txt && tail -n 5 ex5_output.txt

Launching 19532 blocks x 512 threads
9999000) 76 x 52 = 3952
9999996) 40 x 38 = 1520
9999997) 24 x 63 = 1512
9999998) 7 x 69 = 483
9999999) 31 x 73 = 2263
Verification: 0 mismatches out of 10000000


# Exercise 06 — Element-wise Matrix Multiplication

In [ ]:
%%writefile ex6_matrix.cu
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define ROWS 10000
#define COLS 10000

#define CUDA_CHECK(call) do {                                      \
    cudaError_t err = (call);                                      \
    if (err != cudaSuccess) {                                     \
        fprintf(stderr, "CUDA error at line %d: %s\n",             \
                __LINE__, cudaGetErrorString(err));                \
        exit(EXIT_FAILURE);                                       \
    }                                                             \
} while (0)

__global__ void mulElem(const int *a, const int *b, int *c,
                        int rows, int cols)
{
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (row < rows && col < cols) {
        size_t idx = (size_t)row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t count)
{
    for (size_t i = 0; i < count; ++i)
        x[i] = rand() % 100;
}

int main()
{
    size_t count = (size_t)ROWS * COLS;
    size_t bytes = count * sizeof(int);

    int *a = (int *)malloc(bytes);
    int *b = (int *)malloc(bytes);
    int *c = (int *)malloc(bytes);

    if (!a || !b || !c) {
        fprintf(stderr, "Host allocation failed.\n");
        free(a);
        free(b);
        free(c);
        return EXIT_FAILURE;
    }

    random_ints(a, count);
    random_ints(b, count);

    int *d_a, *d_b, *d_c;

    CUDA_CHECK(cudaMalloc((void **)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_c, bytes));

    CUDA_CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));

    dim3 threadsPerBlock(32, 16);
    dim3 numBlocks(
        (COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    printf("Grid: %u x %u blocks\n", numBlocks.x, numBlocks.y);
    printf("Threads per block: %u x %u = %u\n",
           threadsPerBlock.x, threadsPerBlock.y,
           threadsPerBlock.x * threadsPerBlock.y);

    mulElem<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c,
                                            ROWS, COLS);
    CUDA_CHECK(cudaGetLastError());

    CUDA_CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));

    int row = ROWS - 1;
    for (int col = COLS - 1000; col < COLS; ++col) {
        size_t idx = (size_t)row * COLS + col;
        printf("C[%d][%d]: %d x %d = %d\n",
               row, col, a[idx], b[idx], c[idx]);
    }

    size_t errors = 0;
    for (size_t i = 0; i < count; ++i) {
        if (c[i] != a[i] * b[i])
            ++errors;
    }

    printf("Verification: %zu mismatches out of %zu\n",
           errors, count);

    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));

    free(a);
    free(b);
    free(c);

    return errors == 0 ? EXIT_SUCCESS : EXIT_FAILURE;
}

Writing ex6_matrix.cu


In [ ]:
!nvcc -arch=sm_75 ex6_matrix.cu -o ex6_matrix && ./ex6_matrix > ex6_output.txt && head -n 3 ex6_output.txt && tail -n 5 ex6_output.txt

Grid: 313 x 625 blocks
Threads per block: 32 x 16 = 512
C[9999][9000]: 29 x 56 = 1624
C[9999][9996]: 14 x 5 = 70
C[9999][9997]: 69 x 82 = 5658
C[9999][9998]: 9 x 37 = 333
C[9999][9999]: 86 x 21 = 1806
Verification: 0 mismatches out of 100000000
